In [1]:
import requests
import os
import json
import time
import pandas as pd
import sys
import os
import numpy as np
from datetime import date
import modify_votes

In [2]:
try: 
    df1 = pd.read_json("voting_records.json")
except Exception as e:
    print("There is an issue with the voting_records.json. Quitting.")
    sys.exit()

try: 
    df2 = pd.read_json("voting_records_senate.json")
except Exception as e:
    print("There is an issue with the voting_records_senate.json. Quitting.")
    sys.exit()


def merge_house_and_senate

In [22]:
df = modify_votes.merge_house_and_senate(df1, df2)
df['bioguideID'] = np.where(df['lis_member_id'] == "S421", "V000137", df['bioguideID'])
df['bioguideID'] = np.where(df['lis_member_id'] == "S350", "R000595", df['bioguideID'])
# ['S350' is Marco Rubio (R000595), 'S421' is JD Vance (V000137)]
print(f"There are {len(df[df['bioguideID'].isna()])} missing bioguide_ids")
df['chamber'] = np.where(df['lis_member_id'].isna(), "house", "senate") #house is 0, senate is 1

df.head()

0


,congress,identifier,result,voteQuestion,bioguideID,voteCast,voteParty,lis_member_id
0,119,119120251,Passed,Call by States,A000055,Present,R,NaN
1,119,119120251,Passed,Call by States,A000148,Present,D,NaN
2,119,119120251,Passed,Call by States,A000369,Present,R,NaN
3,119,119120251,Passed,Call by States,A000370,Present,D,NaN
4,119,119120251,Passed,Call by States,A000371,Present,D,NaN


Check your numbers here.

In [30]:
all_votes = len(df.groupby('identifier'))
house_vote_count = len(df[df['lis_member_id'].isna()].groupby('identifier'))
senate_vote_count = all_votes - house_vote_count
print(f"{house_vote_count} house votes, {senate_vote_count} senate votes, for total of {all_votes} votes")

total_people = len(df['bioguideID'].unique())
house_people = len(df[df['lis_member_id'].isna()]['bioguideID'].unique())
senate_people = len(df[df['lis_member_id'].notna()]['bioguideID'].unique())
print(f"{house_people} house reps, {senate_people} senators, for total of {total_people} people")

print(len(df))

282 house votes, 530 senate votes, for total of 812 votes
444 house reps, 102 senators, for total of 546 people
175045


In [31]:

#For each vote_identifier, get the count of how each party voted.
#Group by identifier
#
#Add a column that marks if they voted with the party or against the party for that vote
#UPDATE: vote_party is not reliable. changing instead to "voted_with_D" or "voted_with_R"
party_avg_df = df.groupby(['identifier', 'voteParty'])['voteCast'].agg(lambda x: x.mode()[0]).reset_index()

party_mode_votes_wide = party_avg_df.pivot(
    index='identifier',
    columns='voteParty',
    values='voteCast'
).reset_index()

# Rename columns for clarity
party_mode_votes_wide.columns.name = None
party_mode_votes_wide.rename(
    columns={'D': 'D_mode', 'R': 'R_mode'},
    inplace=True
)

# Step 2: Merge the two tables on 'vote_identifier'
# This brings the party mode votes into the individual votes table
merged_df = pd.merge(df, party_mode_votes_wide, on='identifier', how='left')

# Step 3: Create a new column based on a conditional comparison
# Use np.select for clear, readable logic with multiple conditions
conditions = [
    # Voted with both parties (unlikely but possible)
    (merged_df['voteCast'] == merged_df['D_mode']) & (merged_df['voteCast'] == merged_df['R_mode']),
    # Voted with Democrats but not Republicans
    merged_df['voteCast'] == merged_df['D_mode'],
    # Voted with Republicans but not Democrats
    merged_df['voteCast'] == merged_df['R_mode'],
    #Absent
    merged_df['voteCast'] == "Not Voting",
    #Abstained
    merged_df['voteCast'] == "Present"
]

choices = ['Both', 'with_D', 'with_R', 'Absent', 'Abstained']
# Apply the conditions to create the new column
merged_df['voted_with_party'] = np.select(conditions, choices, default='Neither')

merged_df['votecount'] = 1



In [32]:
print(f"These are the options for vote cast: {merged_df['voteCast'].unique()}")
print(len(merged_df))
merged_df

These are the options for vote cast: ['Present' 'Not Voting' 'Jeffries' 'Johnson (LA)' 'Emmer' 'Yea' 'Nay'
 'Aye' 'No']
175045


,congress,identifier,result,voteQuestion,bioguideID,voteCast,voteParty,lis_member_id,chamber,D_mode,I,R_mode,voted_with_party,votecount
0,119,119120251,Passed,Call by States,A000055,Present,R,NaN,house,Present,NaN,Present,Both,1
1,119,119120251,Passed,Call by States,A000148,Present,D,NaN,house,Present,NaN,Present,Both,1
2,119,119120251,Passed,Call by States,A000369,Present,R,NaN,house,Present,NaN,Present,Both,1
3,119,119120251,Passed,Call by States,A000370,Present,D,NaN,house,Present,NaN,Present,Both,1
4,119,119120251,Passed,Call by States,A000371,Present,D,NaN,house,Present,NaN,Present,Both,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
175040,119,530,Nomination Confirmed,On the Nomination,W000800,Nay,D,S422,senate,Nay,Nay,Yea,with_D,1
175041,119,530,Nomination Confirmed,On the Nomination,W000802,Nay,D,S316,senate,Nay,Nay,Yea,with_D,1
175042,119,530,Nomination Confirmed,On the Nomination,W000437,Yea,R,S318,senate,Nay,Nay,Yea,with_R,1
175043,119,530,Nomination Confirmed,On the Nomination,W000779,Nay,D,S247,senate,Nay,Nay,Yea,with_D,1


In [33]:


overall_df = pd.pivot_table(
    merged_df,
    index=['bioguideID', 'chamber'],
    columns='voted_with_party',
    values='votecount',
    aggfunc='sum',
    fill_value=0 # Fills NaN values with 0
).reset_index() # Resets the index to make bioguideID a regular column


print(len(overall_df))

overall_df

546


voted_with_party,bioguideID,chamber,Absent,Abstained,Both,Neither,with_D,with_R
0,A000055,house,9,0,87,0,8,178
1,A000148,house,10,0,89,1,174,8
2,A000369,house,11,0,90,0,9,172
3,A000370,house,0,1,92,1,187,1
4,A000371,house,3,0,91,1,183,4
...,...,...,...,...,...,...,...,...
541,W000830,house,13,0,89,0,157,23
542,W000831,house,0,1,6,0,31,0
543,Y000064,senate,5,0,32,2,2,489
544,Y000067,house,0,0,91,3,1,187


In [35]:
print(len(overall_df))
overall_df['vote_count'] =  overall_df[['Absent', 'Abstained', 'Both', 'Neither', 'with_D', 'with_R']].sum(axis=1)
overall_df['missing_records'] = np.where(overall_df['chamber'] == "house", house_vote_count, senate_vote_count)

overall_df['missing_records'] = overall_df['missing_records'] - overall_df['vote_count']


overall_df.head()


546


voted_with_party,bioguideID,chamber,Absent,Abstained,Both,Neither,with_D,with_R,vote_count,missing_records
0,A000055,house,9,0,87,0,8,178,282,0
1,A000148,house,10,0,89,1,174,8,282,0
2,A000369,house,11,0,90,0,9,172,282,0
3,A000370,house,0,1,92,1,187,1,282,0
4,A000371,house,3,0,91,1,183,4,282,0


In [36]:
overall_df.sort_values(by='vote_count', ascending=True).head()



voted_with_party,bioguideID,chamber,Absent,Abstained,Both,Neither,with_D,with_R,vote_count,missing_records
180,G000578,house,1,0,0,0,0,0,1,281
516,V000137,senate,0,0,1,0,0,0,1,529
411,R000595,senate,0,0,3,0,0,5,8,522
539,W000823,house,9,0,2,0,0,6,17,265
542,W000831,house,0,1,6,0,31,0,38,244


In [37]:
print(f"There are {len(overall_df[overall_df['missing_records']!= 0])} congressmen missing records")
overall_df[overall_df['missing_records']!= 0]




There are 21 congressmen missing records


voted_with_party,bioguideID,chamber,Absent,Abstained,Both,Neither,with_D,with_R,vote_count,missing_records
77,C001078,house,40,0,33,0,63,1,137,145
167,F000484,house,1,0,49,10,1,133,194,88
171,G000551,house,69,0,1,0,1,0,71,211
180,G000578,house,1,0,0,0,0,0,1,281
187,G000590,house,39,0,55,6,0,112,212,70
234,H001103,house,1,0,16,0,24,0,41,241
235,H001104,senate,0,0,28,3,0,491,522,8
244,J000299,house,0,0,44,0,0,147,191,91
253,J000312,senate,18,0,29,1,1,479,528,2
275,K000404,house,10,0,8,0,13,10,41,241


In [16]:
overall_df[overall_df['bioguideID'] == "C001078"]


voted_with_party,bioguideID,lis_member_id,Absent,Abstained,Both,Neither,with_D,with_R,vote_count,missing_records


In [ ]:
#Let's find out which votes are missing for these people:

#C001078, Gerald Connoly, died in 2025. NOT REPLACED.
#G000551, Raul Grijalva, died in 2025. NOT REPLACED.
#T000489, Sylvester Turner, died in 2025. NOT REPLACED.
#G000590, Mark Green, resigned from Congress after the Big Beautiful Bill in July. NOT REPLACED.

#W000823, Michael Waltz, stepped down to serve as national security advisor
#F000484, Randy Fine, replaced Michael Waltz
#G000578, Matt Gaetz, was re-elected but resigned before session could start. His only vote is not present for the 1st vote
#P000622, Jimmy Patronis, replaced Matt Gaetz
#V000137, JD Vance, VP
#R000595, Marco Rubio, stepped down to serve as cabinet


#J000299, Mike Johnson, NOT SURE WHY SOME OF THESE ARE MISSING. Apparently speaker doesn't vote unless they have to, but
# why doesn't he show up as "Not Voting"?

#H001103, Pablo Jose Hernandez, delegate
#K000404, Kimberlyn King-Hinds, delegate
#M001219, James Moylan, is a non-voting member
#N000147 non-voting member
#P000610 non-voting member
#R000600 non-voting member
def check_missing_votes(df):

    acceptable_missing_votes = ["C001078", "G000551", "T000489", "G000590", "W000823", "F000484", "G000578", "P000622", "J000299", 
                                "H001103", "K000404", "M001219", "N000147", "P000610", "R000600", "V000137", "R000595"]

    missing_df = df[df['missing_records']!= 0]
    missing_filtered_df = missing_df[~missing_df['bioguideID'].isin(acceptable_missing_votes)]
    if len(missing_filtered_df) != 0:
        print(f"WARNING: Some congressmen are missing voter information unexpectedly.")
    else:
        df.drop('missing_records', axis=1, inplace=True)

    return df



0


"\nmissing_person = 'J000299'\nall_votes = df['identifier'].unique()\nhave_votes = df[df['bioguideID']==missing_person]['identifier'].unique()\n\nmissing_votes = []\nfor vote in all_votes:\n    if vote not in have_votes:\n        missing_votes.append(vote)\n#Want the difference\nprint(missing_votes)\n"

In [44]:
house_count = len(overall_df[overall_df['chamber']=="house"])

senate_count = len(overall_df[overall_df['chamber']=="senate"])
print(house_count)
print(senate_count)


444
102
